# Eshmun Continued Pretraining

Continues causal LM pretraining on `khairi/eshmun-pretraining` starting from `khairi/Eshmun-125M-Base`.

**Pipeline:**
1. Install Eshmun from GitHub
2. Login to HuggingFace Hub
3. Load model and tokenizer
4. Load dataset (raw — tokenization is deferred to the collator)
5. Define `ProteinCLMCollator` (tokenizes + pads + builds labels on each batch)
6. Configure `TrainingArguments` and `Trainer`
7. Train and push to Hub

## 1. Install dependencies

In [ ]:
!pip install -q git+https://github.com/abidikhairi/eshmun.git
!pip install -q datasets transformers accelerate

## 2. Login to HuggingFace Hub

In [ ]:
from huggingface_hub import login as hf_login

hf_login()  # paste your HF write-access token when prompted

## 3. Imports

In [ ]:
from dataclasses import dataclass
from typing import Any

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configuration

In [ ]:
MODEL_ID   = "khairi/Eshmun-125M-Base"
DATASET_ID = "khairi/eshmun-pretraining"
HF_REPO_ID = "khairi/Eshmun-125M-CPT"   # destination repo on the Hub
OUTPUT_DIR = "/tmp/eshmun-125m-cpt"

MAX_SEQ_LEN = 512

PER_DEVICE_TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4       # effective batch = 32
NUM_TRAIN_EPOCHS            = 1
LEARNING_RATE               = 5e-5
WARMUP_STEPS                = 100
LOGGING_STEPS               = 50
SAVE_STEPS                  = 500

## 5. Model & tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_ID}  ({n_params / 1e6:.1f}M parameters)")

## 6. Dataset

Load the raw dataset without any upfront tokenization — the `ProteinCLMCollator`
handles tokenization, padding, and label construction at batch time, so there is
no slow preprocessing step.

In [ ]:
dataset = load_dataset(DATASET_ID, split="train")

# Keep only the text column the collator expects
dataset = dataset.select_columns(["Content"])

print(dataset)

## 7. Data collator

`ProteinCLMCollator` is called by the `Trainer` data loader on each mini-batch:
- tokenizes the raw `Content` strings with truncation to `max_length`
- pads to the longest sequence in the batch (`padding="longest"`)
- builds `labels` from `input_ids`, masking padding positions with `-100`

In [ ]:
@dataclass
class ProteinCLMCollator:
    tokenizer: PreTrainedTokenizerBase
    max_length: int = 512

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        texts = [f["Content"] for f in features]
        encoded = self.tokenizer(
            texts,
            max_length=self.max_length,
            truncation=True,
            padding="longest",
            return_tensors="pt",
        )
        labels = encoded["input_ids"].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        encoded["labels"] = labels
        return encoded


collator = ProteinCLMCollator(tokenizer=tokenizer, max_length=MAX_SEQ_LEN)

## 8. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id=HF_REPO_ID,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collator,
)

trainer.train()

## 9. Save and push to Hub

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

trainer.push_to_hub(commit_message="continued pretraining checkpoint")
print(f"Model pushed to https://huggingface.co/{HF_REPO_ID}")